# 列ごとの分布は合っているのに、相関の点が低いのはなぜ？

このノートは、有用性の観点 $U_\text{gen}$（一般有用性）の中にある**属性間における順位相関構造の一致（corr）**を扱います（ルールブック §6.1.1）。

**ここで採点する観点**

- 列どうしの順位の関係が、本物と同じか。この関係が壊れたデータで「BMI が高い人ほど TG も高いか」を調べると、本物とは違う答えになります。本物に無かった関係が現れたデータでも同じです。

所要 10 分。

## 0. 準備

練習データ（`participant_data/B_practice.csv`）を使います。

In [1]:
import warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')

# 配布データの場所を探す（リポジトリの配置でも、zip を展開しただけの配置でも動くように）
DATA = next(p for p in (Path('..'), Path('../participant_data'), Path('participant_data'))
            if (p / 'B_practice.csv').exists())

from pwscup2026.kit import selfscore as ss        # CodaBench と同じ採点器

B_ref, REF, CFG, info = ss.load_local_reference(DATA)   # 採点器が使う参照値とパラメータ

# 採点と同じ読み方で読む（float_precision='round_trip'）。既定の読み方だと小数の末尾の桁がずれ、
# 値を変えていない列でも点数がごくわずかに下がることがあります。
B = pd.read_csv(DATA / 'B_practice.csv', float_precision='round_trip')

def score(C: pd.DataFrame) -> dict:
    # C を採点して {utility, facets, detail} を返す（CodaBench と同じ採点器）
    return ss.self_score(C.copy(), B_ref, REF, CFG)

print(f'B: {B.shape[0]} 行 × {B.shape[1]} 列')

B: 1049 行 × 14 列


## 1. 本物の $B$ にある関係

対象は主要9列（連続7列の age・BMI・SBP・TG・HDL・ALT・FPG と、2値の sex・smoking）です。prefecture と `time`・`onset`・`death` は含めません。

Spearman の順位相関は、各列の値を小さい順の順位に置き換えてから相関係数を計算したもので、「一方が大きいほど、もう一方も大きい（小さい）」傾向の強さを −1〜1 で表します。すべての列の組について並べた表を、順位相関行列と呼びます。まず $B$ の順位相関行列を作り、関係の強い列の組を見てみます。

In [2]:
FEATURES = ['age', 'sex', 'BMI', 'SBP', 'TG', 'HDL', 'ALT', 'FPG', 'smoking']
rho_B = B[FEATURES].corr(method='spearman')
iu = np.triu_indices(len(FEATURES), k=1)
pairs = pd.DataFrame({'列の組': [f'{FEATURES[i]} と {FEATURES[j]}' for i, j in zip(*iu)],
                      '順位相関 ρ': rho_B.to_numpy()[iu]})
print(f'列の組の数: {len(pairs)}')
pairs.reindex(pairs['順位相関 ρ'].abs().sort_values(ascending=False).index).head(8).round(3)

列の組の数: 36


,列の組,順位相関 ρ
11,sex と HDL,0.711
27,TG と ALT,0.387
12,sex と ALT,0.387
15,BMI と SBP,0.379
16,BMI と TG,0.355
18,BMI と ALT,0.341
2,age と SBP,0.300
28,TG と FPG,0.299


## 2. 測り方

1. $B$ と $C$ のそれぞれで、9列の Spearman 順位相関行列を作ります。
2. 同じ列どうし（行列の対角）と、並びを入れ替えただけの同じ組を除いた 36 組について、相関係数の絶対差 $d_{ij}=|\rho^C_{ij}-\rho^B_{ij}|$ を求めます。
3. 差の平均 $\bar d$ と、差の 90 パーセンタイル $d_{0.9}$（36 個の差を小さい順に並べて 9 割の位置にある値）を計算します。
4. $\mathrm{corr}_\mathrm{mean}=\operatorname{clip}(1-\bar d/0.40,\,0,\,1)$、$\mathrm{corr}_\mathrm{tail}=\operatorname{clip}(1-d_{0.9}/0.80,\,0,\,1)$ とし、$\mathrm{corr}=\min(\mathrm{corr}_\mathrm{mean},\,\mathrm{corr}_\mathrm{tail})$ です。$\operatorname{clip}(x,0,1)$ は、$x$ が 0 未満なら 0、1 を超えたら 1 にする操作です。

In [3]:
def corr_by_formula(C, B, mean_scale=0.40, tail_scale=0.80, q=0.90):
    # ルールブック §6.1.1 の corr を式どおりに計算する
    rC = C[FEATURES].corr(method='spearman').to_numpy()
    rB = B[FEATURES].corr(method='spearman').to_numpy()
    d = np.abs(rC[iu] - rB[iu])                      # 36 組の相関係数の絶対差
    d_mean, d_q90 = float(d.mean()), float(np.quantile(d, q))
    corr_mean = float(np.clip(1 - d_mean / mean_scale, 0, 1))
    corr_tail = float(np.clip(1 - d_q90 / tail_scale, 0, 1))
    return {'差の平均': d_mean, '差の90パーセンタイル': d_q90,
            'corr_mean': corr_mean, 'corr_tail': corr_tail, 'corr = min': min(corr_mean, corr_tail)}

## 3. 意味

- 列ごとの分布の一致（marginal）は、各列を1本ずつ見るので、列どうしの関係が壊れても下がりません。corr は、その関係を見ます。
- **順位**相関なので、「一方が大きいほど、もう一方も大きい（小さい）」という向きと強さを見ています。値そのものの大きさは見ないので、列ごとに対数を取る、正の値を掛けるといった順序を変えない変換では corr は変わりません。
- 平均 $\bar d$ だけだと、崩れた組が一部に限られるときに 36 組の平均に埋もれます。そのため上位側の差 $d_{0.9}$（差の大きいほうから1割の位置の値）も見て、2つの点数 $\mathrm{corr}_\mathrm{mean}$・$\mathrm{corr}_\mathrm{tail}$ の小さいほうを採ります。
- 本物の関係を**弱めた**場合も、本物に無い関係を**作った**場合も、差の絶対値なので同じように下がります。

## 4. 壊してみる

練習データの $B$ に次の3つの加工をします。

1. TG の値を行のあいだで並べ替える（TG の分布はそのまま、TG とほかの列の関係だけが消える）
2. 6つの検査値（BMI・SBP・TG・HDL・ALT・FPG）を、列ごとに別々に並べ替える
3. TG・ALT・FPG を、それぞれ BMI の順番に並べ直す（本物には無い強い関係を、いくつもの組に作る）

In [4]:
rng = np.random.default_rng(0)
LABS = ['BMI', 'SBP', 'TG', 'HDL', 'ALT', 'FPG']

C1 = B.copy()                                   # 1. TG を並べ替える
C1['TG'] = rng.permutation(C1.TG.to_numpy())

C2 = B.copy()                                   # 2. 6つの検査値を列ごとに並べ替える
for col in LABS:
    C2[col] = rng.permutation(C2[col].to_numpy())

C3 = B.copy()                                   # 3. TG・ALT・FPG を BMI の順番に並べ直す
order = np.argsort(C3.BMI.to_numpy())
for col in ['TG', 'ALT', 'FPG']:
    C3.loc[C3.index[order], col] = np.sort(C3[col].to_numpy())

VARIANTS = {'加工なし': B, '1. TG を並べ替え': C1, '2. 検査値6列を並べ替え': C2, '3. TG・ALT・FPG を BMI の順に': C3}
rows = {}
for name, C in VARIANTS.items():
    o = score(C)
    g = o['detail']['U_gen']
    rows[name] = {'marginal_mean': g['marginal_mean'], **corr_by_formula(C, B),
                  '採点器の corr': g['corr_score'], 'U_gen': o['facets']['U_gen'], 'U': o['utility']}
pd.DataFrame(rows).T.round(4)

,marginal_mean,差の平均,差の90パーセンタイル,corr_mean,corr_tail,corr = min,採点器の corr,U_gen,U
加工なし,1.0,0.0000,0.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
1. TG を並べ替え,1.0,0.0477,0.2564,0.8807,0.6795,0.6795,0.6795,0.6795,0.6795
2. 検査値6列を並べ替え,1.0,0.1821,0.3494,0.5447,0.5632,0.5447,0.5447,0.5447,0.2793
3. TG・ALT・FPG を BMI の順に,1.0,0.1643,0.6521,0.5893,0.1848,0.1848,0.1848,0.1848,0.0000


### どの組が崩れたか

加工1と加工3で、差 $d_{ij}$ が大きかった組を並べます。

In [5]:
def worst_pairs(C, k=5):
    rC = C[FEATURES].corr(method='spearman').to_numpy()
    t = pd.DataFrame({'列の組': [f'{FEATURES[i]} と {FEATURES[j]}' for i, j in zip(*iu)],
                      'B の ρ': rho_B.to_numpy()[iu], 'C の ρ': rC[iu]})
    t['差 d'] = (t['C の ρ'] - t['B の ρ']).abs()
    return t.sort_values('差 d', ascending=False).head(k).round(3)

print('1. TG を並べ替え')
print(worst_pairs(C1).to_string(index=False))
print()
print('3. TG・ALT・FPG を BMI の順に')
print(worst_pairs(C3).to_string(index=False))

1. TG を並べ替え
     列の組  B の ρ  C の ρ   差 d
TG と ALT  0.387  0.014 0.373
BMI と TG  0.355  0.009 0.346
TG と FPG  0.299  0.035 0.264
SBP と TG  0.275  0.014 0.261
sex と TG  0.286  0.034 0.252

3. TG・ALT・FPG を BMI の順に
      列の組  B の ρ  C の ρ   差 d
BMI と FPG  0.203    1.0 0.797
ALT と FPG  0.204    1.0 0.796
 TG と FPG  0.299    1.0 0.701
BMI と ALT  0.341    1.0 0.659
 BMI と TG  0.355    1.0 0.645


### 結果の解釈

- **1.** TG の分布は変わらないので、marginal_mean は 1 のままです。TG を含む組の相関がほぼ 0 になり、差が大きい組では 0.25〜0.37 になりました。36 組の平均（0.048）では小さく見えますが、90 パーセンタイル（0.256）で捉えられ、corr は 0.68 です。この加工では $U_\text{gen}$ も $U$ もこの値で決まっています。
- **2.** 6つの検査値どうしの関係がまとめて消えるので、差の平均も 0.18 まで上がり、corr は 0.54 です。$U$ がさらに低い 0.28 なのは、希少群の有用性（$U_\text{rare}$）がそれより下がったためです。
- **3.** BMI・TG・ALT・FPG のあいだの6組に、本物には無い強い関係（$\rho$ が 1.0）ができました。差の平均は 0.16、90 パーセンタイルは 0.65 で、corr は 0.18 です。$U$ が 0 なのは、この $C$ では Cox 回帰（年齢や検査値などから発症の起こりやすさを表すモデル）の推定が成り立たず、特定タスクでの結果忠実度（$U_\text{spec}$）が 0 点になったためです（ルールブック §6.1.2）。1つの加工が、別の観点で大きく効くことがあります。

## 5. 自分の C で確かめる

corr は、採点結果の `detail` → `U_gen` → `corr_score` です。平均側と上位側のどちらで決まったかは、`detail` → `U_gen` → `correlation` の `mean_score`・`tail_score`（差そのものは `mean_dist`・`tail_dist`）で分かります。

In [6]:
c = score(C1)['detail']['U_gen']['correlation']
{k: round(c[k], 4) for k in ['mean_dist', 'tail_dist', 'mean_score', 'tail_score', 'score']}

{'mean_dist': 0.0477,
 'tail_dist': 0.2564,
 'mean_score': 0.8807,
 'tail_score': 0.6795,
 'score': 0.6795}

自分の `C.csv` を、`starter/` と `participant_data/` が並んでいるフォルダに置き、そのフォルダで次のコマンドを実行すると、自己採点の `--json` の出力で同じ値を見られます。

```sh
cd starter
uv run python -m pwscup2026.kit.selfscore ../C.csv --dist ../participant_data --json
```

Docker を使う場合:

```sh
docker run --rm -v "$PWD":/w hajimeono/pwscup2026-kit:main-process-20260912 score /w/C.csv --dist /w/participant_data --json
```